# ITA105 Lab 3 - Data Normalization

Notebook này xử lý 4 bộ dữ liệu:

- Finance
- Gaming
- Health
- Sports

Nội dung chính:
1. Đọc dữ liệu
2. Kiểm tra shape và missing values
3. Thống kê mô tả
4. Phát hiện ngoại lệ bằng IQR và Z-score
5. Chuẩn hóa Min-Max và Z-Score
6. Vẽ histogram, boxplot, scatterplot
7. So sánh và đưa ra nhận xét


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler

plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATASETS = {
    "Finance": "/mnt/data/ITA105_Lab_3_Finance.csv",
    "Gaming": "/mnt/data/ITA105_Lab_3_Gaming.csv",
    "Health": "/mnt/data/ITA105_Lab_3_Health.csv",
    "Sports": "/mnt/data/ITA105_Lab_3_Sports.csv",
}

OUTPUT_DIR = "/mnt/data/lab3_jupyter_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Hàm hỗ trợ
Các hàm dưới đây sẽ:
- in thông tin cơ bản
- tính IQR và Z-score
- vẽ histogram, boxplot, scatterplot
- tạo dữ liệu Min-Max và Z-Score


In [ ]:
def detect_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower": lower,
        "Upper": upper,
        "Count": int(mask.sum()),
        "Indices": series[mask].index.tolist(),
        "Values": series[mask].tolist(),
    }

def detect_outliers_zscore(series):
    std = series.std(ddof=0)
    if std == 0:
        z = pd.Series(np.zeros(len(series)), index=series.index)
    else:
        z = ((series - series.mean()) / std).abs()
    mask = z > 3
    return {
        "Count": int(mask.sum()),
        "Indices": series[mask].index.tolist(),
        "Values": series[mask].tolist(),
    }

def show_basic_info(name, df):
    print("=" * 90)
    print(f"DATASET: {name}")
    print("Shape:", df.shape)
    print("\nMissing values:")
    display(df.isna().sum().to_frame("missing_values"))

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    desc = df[numeric_cols].describe().T
    desc["median"] = df[numeric_cols].median()
    desc["missing"] = df[numeric_cols].isna().sum()
    display(desc[["mean", "50%", "std", "min", "max", "median", "missing"]])
    return numeric_cols

def plot_hist_box(df, name, numeric_cols):
    for col in numeric_cols:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].hist(df[col], bins=15)
        axes[0].set_title(f"{name} - Histogram - {col}")
        axes[0].set_xlabel(col)
        axes[0].set_ylabel("Frequency")

        axes[1].boxplot(df[col], vert=True)
        axes[1].set_title(f"{name} - Boxplot - {col}")
        axes[1].set_ylabel(col)

        plt.tight_layout()
        plt.show()

def outlier_summary(df, numeric_cols):
    rows = []
    for col in numeric_cols:
        iqr_info = detect_outliers_iqr(df[col])
        z_info = detect_outliers_zscore(df[col])
        rows.append({
            "column": col,
            "iqr_count": iqr_info["Count"],
            "zscore_count": z_info["Count"],
            "iqr_values": iqr_info["Values"],
            "zscore_values": z_info["Values"],
        })
    return pd.DataFrame(rows)

def scale_dataset(df, numeric_cols):
    minmax_scaler = MinMaxScaler()
    zscore_scaler = StandardScaler()

    minmax_df = pd.DataFrame(
        minmax_scaler.fit_transform(df[numeric_cols]),
        columns=numeric_cols
    )

    zscore_df = pd.DataFrame(
        zscore_scaler.fit_transform(df[numeric_cols]),
        columns=numeric_cols
    )
    return minmax_df, zscore_df

def compare_distributions(original_df, minmax_df, zscore_df, name, numeric_cols):
    for col in numeric_cols:
        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        axes[0].hist(original_df[col], bins=15)
        axes[0].set_title("Before")
        axes[1].hist(minmax_df[col], bins=15)
        axes[1].set_title("Min-Max")
        axes[2].hist(zscore_df[col], bins=15)
        axes[2].set_title("Z-Score")
        fig.suptitle(f"{name} - Distribution comparison - {col}")
        plt.tight_layout()
        plt.show()

def save_scaled_files(name, minmax_df, zscore_df):
    minmax_path = os.path.join(OUTPUT_DIR, f"{name.lower()}_minmax.csv")
    zscore_path = os.path.join(OUTPUT_DIR, f"{name.lower()}_zscore.csv")
    minmax_df.to_csv(minmax_path, index=False)
    zscore_df.to_csv(zscore_path, index=False)
    print("Saved:", minmax_path)
    print("Saved:", zscore_path)


## Chạy phân tích cho từng bộ dữ liệu
Cell dưới đây sẽ lặp qua toàn bộ 4 file CSV.


In [ ]:
results = {}

for name, path in DATASETS.items():
    df = pd.read_csv(path)
    numeric_cols = show_basic_info(name, df)

    print("\nOutlier summary:")
    outlier_df = outlier_summary(df, numeric_cols)
    display(outlier_df)

    print("\nBiểu đồ histogram và boxplot:")
    plot_hist_box(df, name, numeric_cols)

    minmax_df, zscore_df = scale_dataset(df, numeric_cols)

    print("\nSo sánh phân phối trước và sau chuẩn hóa:")
    compare_distributions(df, minmax_df, zscore_df, name, numeric_cols)

    save_scaled_files(name, minmax_df, zscore_df)

    results[name] = {
        "raw": df,
        "numeric_cols": numeric_cols,
        "outlier_summary": outlier_df,
        "minmax": minmax_df,
        "zscore": zscore_df,
    }


## Scatterplot theo yêu cầu bài


In [ ]:
# Finance: doanh_thu_musd vs loi_nhuan_musd
finance = results["Finance"]["raw"]
plt.figure(figsize=(6, 4))
plt.scatter(finance["doanh_thu_musd"], finance["loi_nhuan_musd"], alpha=0.7)
plt.xlabel("doanh_thu_musd")
plt.ylabel("loi_nhuan_musd")
plt.title("Finance - doanh_thu_musd vs loi_nhuan_musd")
plt.tight_layout()
plt.show()

# Sports: chieu_cao_cm vs can_nang_kg
sports = results["Sports"]["raw"]
plt.figure(figsize=(6, 4))
plt.scatter(sports["chieu_cao_cm"], sports["can_nang_kg"], alpha=0.7)
plt.xlabel("chieu_cao_cm")
plt.ylabel("can_nang_kg")
plt.title("Sports - chieu_cao_cm vs can_nang_kg")
plt.tight_layout()
plt.show()


## Gợi ý nhận xét
Bạn có thể dùng khung nhận xét ngắn sau cho báo cáo:

### 1. Missing values
- Các dataset không có missing values.

### 2. Outliers
- Finance có outlier rất mạnh ở doanh thu và lợi nhuận.
- Health có outlier ở BMI và huyết áp.
- Sports có outlier ở chiều cao, cân nặng và tốc độ chạy.
- Gaming có outlier ở giờ chơi và điểm tích lũy.

### 3. So sánh Min-Max và Z-Score
- Min-Max phù hợp khi dữ liệu không có quá nhiều outlier lớn.
- Khi có outlier mạnh, Min-Max dễ làm dữ liệu còn lại bị nén sát về 0.
- Z-Score thường phù hợp hơn cho các bộ dữ liệu trong lab này.

### 4. Ảnh hưởng đến mô hình
- KNN, K-Means, SVM thường cần chuẩn hóa.
- Với dữ liệu có ngoại lệ mạnh, Z-Score thường ổn định hơn.


## Lưu ý
- File chuẩn hóa sẽ được lưu trong thư mục: `/mnt/data/lab3_jupyter_outputs`
- Bạn có thể tách cell lớn thành từng dataset nếu muốn trình bày riêng từng bài.
